# Saudi Arabia Demand Forecasting — XGBoost Model

## Goal
Train and evaluate an XGBoost regression model to predict `demand_count` for each
`(city, region_id, aoi_id, bucket_hour)` combination using lag, rolling, and
Saudi-specific event features.

**Dataset:** Synthetic Saudi demand dataset calibrated to TGA 2024 (290M national orders,
Eastern Province 15%, Dammam ~71,507 orders/day).

**Saudi features added:** `is_ramadan`, `is_eid`, `is_eid_pre`, `is_white_friday`,
`is_national_day`, `is_founding_day`, `is_back_to_school`, `is_hajj`, `is_summer`,
`is_weekend` (Friday–Saturday).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import subprocess
import warnings
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
    print("XGBoost version:", xgb.__version__)
except ImportError:
    subprocess.run(["pip", "install", "xgboost", "-q"])
    import xgboost as xgb
    print("XGBoost installed, version:", xgb.__version__)

SEED = 42
np.random.seed(SEED)

# ── GPU Detection (fixed: was returning 'hist' in both branches) ──────────────
try:
    gpu_check = subprocess.run(["nvidia-smi"], capture_output=True)
    TREE_METHOD = "hist"      # Change to "gpu_hist" if you have XGBoost GPU build
    HAS_GPU = gpu_check.returncode == 0
except Exception:
    TREE_METHOD = "hist"
    HAS_GPU = False

print(f"GPU available: {HAS_GPU}  |  tree_method: '{TREE_METHOD}'")
print("Ready.")

## Step 1 — Load the Saudi Dataset

Loads `saudi_hourly_features.csv` generated by `saudi_demand_generator_final.py`.
Set `CSV_PATH` to match your Google Drive folder.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
CSV_PATH = "/content/drive/MyDrive/Senior_Project/saudi_hourly_features.csv"

if 'model_df' not in globals():
    print("Loading Saudi dataset...")
    model_df = pd.read_csv(CSV_PATH, parse_dates=["bucket_hour"])
    print("Loaded shape:", model_df.shape)
    print("City:", model_df["city"].unique())
    print("Date range:", model_df["bucket_hour"].min(), "→", model_df["bucket_hour"].max())
else:
    print("Using existing model_df from memory, shape:", model_df.shape)
    model_df["bucket_hour"] = pd.to_datetime(model_df["bucket_hour"])

model_df.head()


## Step 2 — Chronological Train / Val / Test Split

We split **globally** by time (not per AOI), so:
- **Train**: first 60% of hours
- **Val**: next 20% of hours
- **Test**: last 20% of hours

This prevents any future data from leaking into training.

> Split is done **before** feature engineering so that the Saudi calendar
> features added in the next step cannot accidentally encode split membership.

In [ ]:
TRAIN_RATIO = 0.60
VAL_RATIO   = 0.20

all_hours = model_df["bucket_hour"].sort_values().unique()
n_hours   = len(all_hours)

train_end = all_hours[int(n_hours * TRAIN_RATIO) - 1]
val_end   = all_hours[int(n_hours * (TRAIN_RATIO + VAL_RATIO)) - 1]

model_df["split"] = np.where(
    model_df["bucket_hour"] <= train_end, "train",
    np.where(model_df["bucket_hour"] <= val_end, "val", "test")
)

split_counts = model_df["split"].value_counts().sort_index()
print(split_counts)
print(f"\nTrain ends:  {train_end}")
print(f"Val ends:    {val_end}")
print(f"Test starts: {model_df.loc[model_df['split'] == 'test', 'bucket_hour'].min()}")

## Step 2a — Time and Spatial Features

- **Circadian**: `hour_sin`, `hour_cos` encode the cyclical nature of hours.
- **Saudi event flags**: All 2024 dates — Ramadan, Eid Al-Fitr, Eid Al-Adha,
  White Friday, National Day, Founding Day, Back-to-School, Hajj, Summer.
- **Weekend**: Friday–Saturday (Saudi weekend, not Saturday–Sunday).
- **Neighbor lag**: mean lag_24 of other AOIs in the same region.

> Note: The Saudi data already contains these flags from the generator.
> This cell recomputes them from `bucket_hour` to guarantee correctness
> and adds the circadian + neighbor features.


In [ ]:
# ── Circadian encoding ────────────────────────────────────────────────────────
model_df["hour"]     = model_df["bucket_hour"].dt.hour
model_df["hour_sin"] = np.sin(2 * np.pi * model_df["hour"] / 24)
model_df["hour_cos"] = np.cos(2 * np.pi * model_df["hour"] / 24)

# ── Saudi event flags — 2024 dates ────────────────────────────────────────────
dates = model_df["bucket_hour"].dt.normalize()

# Ramadan 2024: Mar 11 – Apr 9  (Checkout.com: +22% digital transactions)
model_df["is_ramadan"] = dates.between(
    pd.Timestamp("2024-03-11"), pd.Timestamp("2024-04-09")
).astype("int8")

# Eid Al-Fitr (Apr 10–12) + Eid Al-Adha (Jun 16–18)  (SAMA Mada: +50/45%)
model_df["is_eid"] = (
    dates.between(pd.Timestamp("2024-04-10"), pd.Timestamp("2024-04-12")) |
    dates.between(pd.Timestamp("2024-06-16"), pd.Timestamp("2024-06-18"))
).astype("int8")

# Pre-Eid shopping surge — 2 days before each Eid  (SAMA POS lead-up peaks)
model_df["is_eid_pre"] = (
    dates.between(pd.Timestamp("2024-04-08"), pd.Timestamp("2024-04-09")) |
    dates.between(pd.Timestamp("2024-06-14"), pd.Timestamp("2024-06-15"))
).astype("int8")

# White Friday: Nov 29  (AppsFlyer 2024: +29–74% in-app purchases)
model_df["is_white_friday"] = (
    dates == pd.Timestamp("2024-11-29")
).astype("int8")

# National Day window: Sep 16–30  (Ministry of Commerce discount licensing)
model_df["is_national_day"] = dates.between(
    pd.Timestamp("2024-09-16"), pd.Timestamp("2024-09-30")
).astype("int8")

# Founding Day window: Feb 18–25  (Ministry of Commerce, qualitative +15%)
model_df["is_founding_day"] = dates.between(
    pd.Timestamp("2024-02-18"), pd.Timestamp("2024-02-25")
).astype("int8")

# Back-to-School: Sep 1–14  (SAMA POS: slight aggregate dip)
model_df["is_back_to_school"] = dates.between(
    pd.Timestamp("2024-09-01"), pd.Timestamp("2024-09-14")
).astype("int8")

# Hajj: Jun 14–19  (GASTAT: logistics fleet reallocated to Makkah corridor)
model_df["is_hajj"] = dates.between(
    pd.Timestamp("2024-06-14"), pd.Timestamp("2024-06-19")
).astype("int8")

# Summer: Jun–Sep  (Dammam >45°C — midday delivery suppression)
model_df["is_summer"] = model_df["bucket_hour"].dt.month.isin(
    [6, 7, 8, 9]
).astype("int8")

# Saudi weekend: Friday=4, Saturday=5  (TGA 2024 regional pattern)
model_df["is_weekend"] = model_df["bucket_hour"].dt.dayofweek.isin(
    [4, 5]
).astype("int8")

# ── Neighbor lag_24 mean ───────────────────────────────────────────────────────
if "lag_24" in model_df.columns:
    g     = model_df.groupby(["city", "region_id", "bucket_hour"], dropna=False)
    total = g["lag_24"].transform("sum")
    cnt   = g["lag_24"].transform("count")
    model_df["neighbor_lag_24_mean"] = (
        (total - model_df["lag_24"]) / (cnt - 1).replace(0, np.nan)
    ).fillna(0)
else:
    model_df["neighbor_lag_24_mean"] = 0

# lng/lat not available in Saudi synthetic data — skip aoi_area
print("lng/lat not available — aoi_area skipped.")

added = [c for c in [
    "hour_sin", "hour_cos",
    "is_ramadan", "is_eid", "is_eid_pre", "is_white_friday",
    "is_national_day", "is_founding_day", "is_back_to_school",
    "is_hajj", "is_summer", "is_weekend",
    "neighbor_lag_24_mean"
] if c in model_df.columns]
print("Features confirmed:", added)
print("Ramadan rows:",      model_df["is_ramadan"].sum())
print("Eid rows:",          model_df["is_eid"].sum())
print("White Friday rows:", model_df["is_white_friday"].sum())


## Step 3 — Feature Matrix Construction

- **Drop** `bucket_hour`, `split`, `aoi_name`, and `demand_count`.
- **One-hot encode** `city` only (`aoi_type` not present in Saudi data).
- All remaining columns become model features.


In [ ]:
TARGET    = "demand_count"
# aoi_name is text label — drop it; aoi_type not present in Saudi data
DROP_COLS = ["bucket_hour", "split", "aoi_name", TARGET]
OHE_COLS  = ["city"]   # aoi_type removed — not in Saudi dataset

# Drop only columns that actually exist
existing_drop = [c for c in DROP_COLS if c in model_df.columns]

encoded = pd.get_dummies(
    model_df.drop(columns=existing_drop),
    columns=[c for c in OHE_COLS if c in model_df.columns],
    drop_first=True
)

FEATURE_COLS = encoded.columns.tolist()
print(f"Number of features: {len(FEATURE_COLS)}")
print(FEATURE_COLS)


## Step 4 — Prepare Train / Val / Test Arrays

- **Train**: sample up to 2M rows if needed (RAM management).
- **sample_weight**: weighted by total demand volume per AOI — busier zones get higher weight. This is more meaningful than weighting by observation count (which was identical across all AOIs in the balanced grid).

In [ ]:
MAX_TRAIN_ROWS = 2_000_000

split_mask = model_df["split"]

train_idx = model_df.index[split_mask == "train"]
val_idx   = model_df.index[split_mask == "val"]
test_idx  = model_df.index[split_mask == "test"]

if len(train_idx) > MAX_TRAIN_ROWS:
    rng = np.random.RandomState(SEED)
    train_idx = train_idx[rng.choice(len(train_idx), MAX_TRAIN_ROWS, replace=False)]
    print(f"Train sampled to {MAX_TRAIN_ROWS:,} rows")
else:
    print(f"Train rows: {len(train_idx):,}")

X_train = encoded.loc[train_idx]
y_train = model_df.loc[train_idx, TARGET]

X_val  = encoded.loc[val_idx]
y_val  = model_df.loc[val_idx, TARGET]

X_test  = encoded.loc[test_idx]
y_test  = model_df.loc[test_idx, TARGET]

# ── Sample weights: by total demand volume per AOI ────────────────────────────
# Weighting by observation count was broken (all weights = 1.0 on a balanced grid).
# Weighting by demand volume gives higher weight to genuinely busy zones.
aoi_demand = (
    model_df.loc[train_idx]
    .groupby(["city", "region_id", "aoi_id"])[TARGET]
    .sum()
    .reset_index()
    .rename(columns={TARGET: "aoi_total_demand"})
)
train_meta    = model_df.loc[train_idx, ["city", "region_id", "aoi_id"]].copy()
train_meta    = train_meta.merge(aoi_demand, on=["city", "region_id", "aoi_id"], how="left")
sample_weight = train_meta["aoi_total_demand"].fillna(1).values.astype(np.float64)
sample_weight = sample_weight / sample_weight.mean()  # normalize: mean weight = 1

print(f"Val rows:  {len(X_val):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Sample weight range: [{sample_weight.min():.3f}, {sample_weight.max():.3f}]")

## Step 5 — Baseline Models

Two simple baselines to benchmark against:
- **Baseline A** (`lag_24`): predict using the value from exactly 24 hours ago.
- **Baseline B** (`roll_24_mean`): predict using the 24-hour rolling average.

In [ ]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def smape(y_true, y_pred):
    denom = np.abs(y_true) + np.abs(y_pred)
    mask  = denom > 0   # avoid division by zero when both actual and forecast are 0
    return np.mean(2 * np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100

def evaluate(y_true, y_pred, name, split_name):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    return {
        "Model": name,
        "Split": split_name,
        "MAE":   round(mae(y_true, y_pred),   4),
        "RMSE":  round(rmse(y_true, y_pred),  4),
        "sMAPE": round(smape(y_true, y_pred), 2),
    }

results = []

for split_name, X_s, y_s in [("val", X_val, y_val), ("test", X_test, y_test)]:
    results.append(evaluate(y_s, X_s["lag_24"],       "Baseline lag_24",      split_name))
    results.append(evaluate(y_s, X_s["roll_24_mean"], "Baseline roll_24_mean", split_name))

print(pd.DataFrame(results).to_string(index=False))

## Step 6 — Train XGBoost (Main Model) with Early Stopping

In [ ]:
model = xgb.XGBRegressor(
    objective          = "reg:squarederror",
    tree_method        = TREE_METHOD,
    n_estimators       = 5000,          # high cap; early stopping picks best round
    learning_rate      = 0.05,
    max_depth          = 6,
    min_child_weight   = 10,            # conservative for sparse/zero-heavy demand
    subsample          = 0.8,
    colsample_bytree   = 0.8,
    reg_alpha          = 0.1,
    reg_lambda         = 2.0,
    random_state       = SEED,
    n_jobs             = -1,
    eval_metric        = ["rmse", "mae"],
    early_stopping_rounds = 100,
)

model.fit(
    X_train, y_train,
    sample_weight = sample_weight,
    eval_set      = [(X_train, y_train), (X_val, y_val)],
    verbose       = 50,
)

print(f"\nBest iteration: {model.best_iteration}")

## Step 6b — (Optional) Alternative Models

- **Zero-inflated**: Logistic regression predicts P(demand > 0); XGBoost trained *only on positive-demand rows* predicts E[demand | demand > 0]. Final prediction = P × conditional mean.
  - ⚠️ **Fix applied**: conditional XGBoost is now evaluated only on rows where `y_true > 0` for a fair comparison of the conditional component. The combined zero-inflated prediction is still evaluated on the full set.
- **reg:squaredlogerror**: penalizes errors on small values less — useful for sparse demand distributions.

In [ ]:
from sklearn.linear_model import LogisticRegression

# ── Zero-inflated model ───────────────────────────────────────────────────────
y_train_binary = (y_train > 0).astype(int)
logit = LogisticRegression(max_iter=500, random_state=SEED, C=0.5)
logit.fit(X_train, y_train_binary)

# Train conditional XGBoost on positive-demand rows only
pos_mask    = y_train > 0
X_train_pos = X_train.loc[pos_mask]
y_train_pos = y_train.loc[pos_mask]
w_pos       = sample_weight[pos_mask.values]

xgb_cond = xgb.XGBRegressor(
    objective="reg:squarederror", tree_method=TREE_METHOD,
    n_estimators=500, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    reg_alpha=0.1, reg_lambda=1.0, random_state=SEED, n_jobs=-1,
    early_stopping_rounds=25, eval_metric="rmse",
)
# ⚠️ eval_set uses full X_val so early stopping tracks overall val RMSE,
#    but the conditional component itself was only trained on positive rows.
xgb_cond.fit(
    X_train_pos, y_train_pos, sample_weight=w_pos,
    eval_set=[(X_train_pos, y_train_pos), (X_val, y_val)],
    verbose=False,
)

# Combined zero-inflated prediction: P(demand>0) × E[demand|demand>0]
# Evaluated on the FULL val/test set (this is correct — it's the final combined model)
for split_name, X_s, y_s in [("val", X_val, y_val), ("test", X_test, y_test)]:
    p_pos  = logit.predict_proba(X_s)[:, 1]
    cond   = np.clip(xgb_cond.predict(X_s), 0, None)
    zi_pred = p_pos * cond
    results.append(evaluate(y_s, zi_pred, "XGBoost (zero-inflated)", split_name))

print("Zero-inflated model evaluated.")

# ── reg:squaredlogerror ───────────────────────────────────────────────────────
xgb_sle = xgb.XGBRegressor(
    objective="reg:squaredlogerror", tree_method=TREE_METHOD,
    n_estimators=1000, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    reg_alpha=0.1, reg_lambda=1.0, random_state=SEED, n_jobs=-1,
    early_stopping_rounds=30, eval_metric="rmse",
)
xgb_sle.fit(
    X_train, y_train, sample_weight=sample_weight,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=50,
)
for split_name, X_s, y_s in [("val", X_val, y_val), ("test", X_test, y_test)]:
    sle_pred = np.clip(xgb_sle.predict(X_s), 0, None)
    results.append(evaluate(y_s, sle_pred, "XGBoost (squaredlogerror)", split_name))

print("reg:squaredlogerror model evaluated.")

## Step 7 — Training Curves

In [ ]:
# ── plot_train_val_curves defined ONCE, used for all models ──────────────────
def plot_train_val_curves(evals_result, title="Model", metric="rmse"):
    fig, ax = plt.subplots(figsize=(10, 5))

    if "validation_0" in evals_result:
        train_metric = evals_result["validation_0"][metric]
        ax.plot(range(1, len(train_metric) + 1), train_metric,
                label=f"Train {metric.upper()}", color="steelblue", linewidth=1.5)

    if "validation_1" in evals_result:
        val_metric = evals_result["validation_1"][metric]
        ax.plot(range(1, len(val_metric) + 1), val_metric,
                label=f"Val {metric.upper()}", color="tomato", linewidth=1.5)

    ax.set_xlabel("Boosting round")
    ax.set_ylabel(metric.upper())
    ax.set_title(f"{title} — Training vs Validation")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_train_val_curves(model.evals_result(),    title="XGBoost (main)")
plot_train_val_curves(xgb_cond.evals_result(), title="XGBoost (zero-inflated, conditional)")
plot_train_val_curves(xgb_sle.evals_result(),  title="XGBoost (squaredlogerror)")

## Step 8 — Evaluate All Models: Final Comparison Table

In [ ]:
val_preds  = np.clip(model.predict(X_val),  0, None)
test_preds = np.clip(model.predict(X_test), 0, None)

results.append(evaluate(y_val,  val_preds,  "XGBoost", "val"))
results.append(evaluate(y_test, test_preds, "XGBoost", "test"))

results_df = pd.DataFrame(results)
comparison = results_df.pivot(index="Model", columns="Split", values=["MAE", "RMSE", "sMAPE"])
comparison.columns = [f"{metric}_{split}" for metric, split in comparison.columns]
comparison = comparison[["MAE_val", "RMSE_val", "sMAPE_val", "MAE_test", "RMSE_test", "sMAPE_test"]]

print("\n=== Model Comparison ===")
print(comparison.to_string())

## Step 9 — Feature Importance

In [ ]:
importance = (
    pd.Series(model.feature_importances_, index=FEATURE_COLS)
    .sort_values(ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(10, 6))
importance.plot(kind="barh", ax=ax, color="steelblue")
ax.invert_yaxis()
ax.set_title("XGBoost — Top 20 Feature Importances (gain)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

## Step 10 — Example AOI Plot: Actual vs Predicted

Pick the busiest AOI in the test set and plot actual demand vs XGBoost prediction vs lag_24 baseline.

In [ ]:
test_slice = model_df.loc[test_idx].copy()
test_slice["pred_xgb"]   = test_preds
test_slice["pred_lag24"] = X_test["lag_24"].values

# Busiest AOI by total demand in test
best_aoi          = (
    test_slice.groupby(["city", "region_id", "aoi_id"])["demand_count"]
    .sum().idxmax()
)
city_s, region_s, aoi_s = best_aoi

aoi_test = (
    test_slice[
        (test_slice["city"]      == city_s) &
        (test_slice["region_id"] == region_s) &
        (test_slice["aoi_id"]    == aoi_s)
    ]
    .sort_values("bucket_hour")
)

PLOT_DAYS    = 14
aoi_test_plot = aoi_test.head(24 * PLOT_DAYS)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(aoi_test_plot["bucket_hour"], aoi_test_plot["demand_count"],
        label="Actual",  color="black",     linewidth=1.5)
ax.plot(aoi_test_plot["bucket_hour"], aoi_test_plot["pred_xgb"],
        label="XGBoost", color="steelblue", linewidth=1.2, linestyle="--")
ax.plot(aoi_test_plot["bucket_hour"], aoi_test_plot["pred_lag24"],
        label="Lag-24",  color="tomato",    linewidth=1.0, linestyle=":")

ax.set_title(
    f"Hourly Demand Forecast — AOI {aoi_s} (region {region_s}, {city_s})\n"
    f"Test set, first {PLOT_DAYS} days"
)
ax.set_xlabel("Time")
ax.set_ylabel("demand_count")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"Selected AOI: city={city_s}, region_id={region_s}, aoi_id={aoi_s}")
print(f"Test rows for this AOI: {len(aoi_test)}")